In [1]:
from models import *
import torch
from dataloaders import *
import torch.nn as nn
from torch.utils.data import DataLoader
from shenghao_data import *

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [6]:
model = SimpleTransformerModel(d_model=64, n_heads=4, num_layers=4, tropical=True, num_classes=64, activation='relu', 
                           tropical_attention_cls = TropicalAttention(64, 4, torch.device('cuda')), pool=True, skip=True).to(device)

In [7]:
ckpt_path = '15_exp/models/FloydWarshallDataset_tropical_0.0001_20000_20260403_100456_relu_best.pth'
    
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [8]:
num_train_samples = 50000
num_val_samples = 10000
n = 8
low_train = 1
high_train = 15
low_test = 1
high_test = 15
use_integer = True
num_additional_node = 0
batch_size = 1
shuffle = True

In [9]:
val_dataset = FloydWarshallDataset(num_val_samples, adversarial_range=(10, 20), length_range=(8, 8), noise_prob=0, value_range=(1, 15))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [22]:
np.set_printoptions(precision=2, suppress=True)
criterion = nn.MSELoss()
model.eval()
with torch.no_grad():
    val_loss = 0
    for (x, y) in val_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        for i in range(y.size(0)):
            print(f'y[{i}]:')
            print(y[i].view(8, 8).cpu().numpy())
            print()
            print(f'out[{i}]:')
            print(out[i].view(8, 8).cpu().numpy())
            print()
        val_loss += criterion(torch.triu(out, 1), torch.triu(y, 1)).item()
        print(val_loss)
        break

y[0]:
[[8 7 4 2 0 4 4 4]
 [4 8 5 2 7 1 1 1]
 [4 5 8 2 2 2 5 5]
 [4 5 3 8 2 2 3 3]
 [4 7 4 2 8 4 4 4]
 [4 5 5 2 5 8 5 5]
 [4 6 5 6 6 6 8 5]
 [4 7 5 7 7 7 5 8]]

out[0]:
[[ 0.    0.81  0.47  0.67  0.14  0.73  0.46  0.65]
 [ 0.81  0.    0.5   0.62  0.66  0.13  0.4   0.16]
 [ 0.47  0.5  -0.    0.11  0.38  0.36  0.66  0.55]
 [ 0.67  0.62  0.11  0.    0.47  0.46  0.73  0.53]
 [ 0.14  0.66  0.38  0.47  0.    0.57  0.35  0.55]
 [ 0.73  0.13  0.36  0.46  0.57  0.    0.32  0.21]
 [ 0.46  0.4   0.66  0.73  0.35  0.32 -0.    0.53]
 [ 0.65  0.16  0.55  0.53  0.55  0.21  0.53  0.  ]]

21.511234283447266


In [11]:
print(val_loss / len(val_loader))

19.65925722732544
